# Slowly Changing Dimensions (SCD)

In dimensional modeling, a **slowly changing dimension** is a dimension whose attribute values change over time — slowly, not every transaction. Ralph Kimball defined six classic types (0–6); a seventh is commonly added in practice. Each type answers the question: *when an attribute value changes, what do we do with the old value?*

---

| Type | Name | Strategy | Year announced | Old value preserved? | Full history? | New objects created |
|------|------|---------|---------------|---------------------|---------------|--------------------|
| 0 | Retain / Original | Never change the value | 1996 (2008*) | ✅ (never overwritten) | ❌ | None — no schema or table changes |
| 1 | Overwrite | Replace old with new | 1996 | ❌ | ❌ | None — same table, same columns, in-place updates |
| 2 | Add New Row | Insert a new row, close out the old | 1996 | ✅ | ✅ | **New columns** added to the existing dimension: `valid_from`, `valid_to`, `is_current`, plus a **surrogate key** column (e.g. `product_sk`) |
| 3 | Add New Column | Keep old + new in separate columns | 1996 | ✅ (one prior version) | Partial (current + previous) | **New columns** added to the existing dimension (e.g. `previous_category`) — one per tracked attribute |
| 4 | Mini-Dimension | Rapidly changing attrs split into a separate dimension | 2002 | N/A (separate table) | ✅ (via mini-dim grouping) | **New table** (mini-dimension) for the rapidly-changing attributes, keyed by a surrogate key; base dimension gains a **new FK column** (e.g. `mini_sk`) pointing to it |
| 5 | Mini-Dimension + Type 1 Outrigger | Type 4 + Type 1 ref in base dim | 2002 | Partial | ✅ (via mini-dim) | Everything from Type 4 **plus new outrigger columns** copied into the base dimension (e.g. `current_category`, `current_brand`) carrying the latest mini-dim values |
| 6 | Hybrid (1 + 2) | Type 2 history + Type 1 overwrite for selected attrs | 2002 | Partial | ✅ | All Type 2 columns **plus a new current-value column** (e.g. `current_category`) stamped across every history row |
| 7 | Dual Type (1 + 2) | Maintain both a Type 1 table and a Type 2 table | 2013 | Partial | ✅ | **Second physical table** — a Type 1 snapshot table alongside the Type 2 history table (two tables maintained in parallel) |

---

*announced in 1996 with other concepts, but described in details in 2008.

## Detailed explanations

### Type 0 — Retain (Original Value)
Never modify the attribute — the value set at insert time is permanent, regardless of later changes in the source. Rarely used, but useful for immutable facts like a customer's original registration date or date of birth.

> **New objects:** None. The dimension table and its schema stay exactly as-is; no columns, tables, or lookup structures are added.
>
> **Announced:** Formally described by the Kimball Group in the third edition of *The Data Warehouse Toolkit* (2008), complementing the original three types.
> **Data Warehouse use case:** Preserving immutable attributes such as a customer's original registration date or date of birth that must never be modified.
> **Data Lakehouse use case:** Retaining original source-system landing timestamps in the Bronze layer for full audit traceability across the medallion architecture.

### Type 1 — Overwrite
Simply update the row in place. No history is kept — the old value is lost. This is the simplest and most common approach when only the current value matters (e.g., correcting a misspelled name).

> **New objects:** None. Same table, same columns — rows are simply overwritten in place.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Correcting a misspelled customer name or keeping the latest shipping address when only the current value matters.
> **Data Lakehouse use case:** Overwriting the latest record state in the Silver/Gold Delta tables via `MERGE` for downstream consumers that need only current values.

### Type 2 — Add New Row (Historical Tracking)
Insert a **new row** for the changed record and mark the previous row as expired. This preserves the full history of changes. Each row gets:
* **effective dates** (or timestamps): `valid_from` / `valid_to`
* a **current flag** (`is_current` = true/false)
* a **surrogate key** so every version has a unique primary key

This is the most widely used SCD type in data warehousing because it lets fact rows join to the dimension version that was active at the time of the fact.

> **New objects:** The **existing dimension table gains new columns** — at minimum a surrogate key (e.g. `product_sk`), `valid_from`, `valid_to`, and `is_current`. No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Tracking full customer address history so that fact rows join to the dimension version that was active at the time of each transaction.
> **Data Lakehouse use case:** Maintaining full version history in Delta tables using Change Data Feed (CDF) with `MERGE` SCD Type 2 logic for point-in-time analytics and incremental CDC pipelines.

### Type 3 — Add New Column
Add a second column to hold the prior value (e.g., `previous_category` alongside `category`). Only one level of history is kept, so this is suitable when you only need to compare the current value to the immediately preceding one.

> **New objects:** **New columns** added to the existing dimension table — one `previous_*` column per tracked attribute (e.g. `previous_category`). No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Comparing current vs. previous sales-territory assignment for an employee without the overhead of full history rows.
> **Data Lakehouse use case:** Lightweight one-level history tracking in a Silver Delta table when full SCD Type 2 is unnecessary but a before/after comparison is still required.

### Type 4 — Mini-Dimension
When certain attributes change **rapidly** (e.g., number of dependents, income band, credit score), they are split out of the main dimension into a separate *mini-dimension*. Each unique combination of those attributes becomes a row in the mini-dimension, keyed by a surrogate key. The base dimension references the current mini-dimension key (a Type 1 overwrite on that key). Fact rows then reference the mini-dimension surrogate key, capturing history without exploding the base dimension.

> **New objects:** A **new lookup table** (the mini-dimension) holding one row per unique attribute combination, keyed by its own surrogate key. The base dimension gains a **new foreign-key column** (e.g. `mini_sk`) referencing the mini-dimension.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) to handle rapidly changing attributes.
> **Data Warehouse use case:** Managing fast-changing customer demographics (income band, credit score, dependents) without exploding the base customer dimension.
> **Data Lakehouse use case:** Splitting volatile attributes into a separate Delta table keyed by surrogate key, referenced from the main dimension to keep the base table compact and queryable.

### Type 5 — Mini-Dimension + Type 1 Outrigger
A Type 4 mini-dimension, plus the **base dimension also carries a Type 1 copy** of the key mini-dimension attributes as an outrigger (overwritten on change). This lets you group or filter by the *current* value of a rapidly changing attribute directly from the base dimension, while the mini-dimension preserves the full history.

> **New objects:** Everything from Type 4 (new mini-dimension table + `mini_sk` FK on the base) **plus new outrigger columns** on the base dimension (e.g. `current_category`, `current_brand`) that carry a Type 1 copy of the latest mini-dimension attribute values.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as an extension of the Type 4 mini-dimension pattern.
> **Data Warehouse use case:** Allowing BI dashboards to filter by the current income band directly from the base dimension while the mini-dimension preserves full historical accuracy.
> **Data Lakehouse use case:** Adding current-value outrigger columns in the Gold layer so BI tools can query volatile attributes without joining to the mini-dimension Delta table.

### Type 6 — Hybrid (Type 1 + Type 2)
Combines Type 2 (full history rows with effective dates) **and** Type 1 (overwrite) for a subset of columns. Typical implementation: the dimension has history rows (Type 2), but also maintains a **current-value column** that is always overwritten to the latest value (Type 1) across *all* rows. This lets you group all historical facts by the current value of an attribute, while still preserving the value as it was at each point in time.

> **New objects:** All Type 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`) **plus a new current-value column** (e.g. `current_category`) added to the existing dimension table. No additional tables are created.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as a hybrid combining Types 1 and 2.
> **Data Warehouse use case:** Grouping all historical sales by the *current* product category while still preserving the category value as it was at each point in time.
> **Data Lakehouse use case:** Stamping a `current_*` column across every history row in a Delta table so a single Gold-layer query supports both current and point-in-time analytics.

### Type 7 — Dual Type 1 + Type 2
Maintain **two physical tables**: one Type 1 table (current state only, overwritten) and one Type 2 table (full history). Fact tables join to the Type 2 table for point-in-time accuracy, while the Type 1 table serves dashboards and queries that only need the latest snapshot. This trades extra storage for query simplicity.

> **New objects:** A **second physical table** — a Type 1 snapshot table alongside the Type 2 history table. The Type 2 table carries the SCD 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`); the Type 1 table mirrors the original dimension schema. Two tables must be maintained and kept in sync.
>
> **Announced:** Type 7 was formally described in 2013 by the Kimball Group. It was included in the 3rd edition of The Data Warehouse Toolkit the same year.
> **Data Warehouse use case:** Serving a Type 1 current-state table for fast dashboards and a Type 2 history table for accurate point-in-time fact joins from the same source feed.
> **Data Lakehouse use case:** Maintaining a current-state Delta table (feeding BI/SQL warehouses) alongside a full-history Delta table (feeding incremental CDC and audit pipelines), both refreshed from the same batch or streaming source.





## Type 2 example (most common in practice)

Using the `dim_product` table above, if product 3 changes from category *Kitchen* to *Home*, a Type 2 table would look like:

| product_sk (surrogate) | product_id | product_name | category | valid_from | valid_to | is_current |
|------------------------|------------|--------------|----------|------------|----------|------------|
| 301 | 3 | Coffee Mug | Kitchen | 2024-01-01 | 2024-06-30 | false |
| 302 | 3 | Coffee Mug | Home | 2024-07-01 | 9999-12-31 | true |

The surrogate key `product_sk` is unique per version; the natural key `product_id` repeats. Fact rows reference `product_sk` so each fact is associated with the correct version of the product.

> **Reference:** [Slowly changing dimension — Wikipedia](https://en.wikipedia.org/wiki/Slowly_changing_dimension)

## 2. Sample Dimension table

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dim_product = spark.createDataFrame([
    Row(product_id=1, product_name="Laptop",    category="Electronics", brand="Acme",     color="Silver"),
    Row(product_id=2, product_name="Headphones", category="Electronics", brand="Bose",     color="Black"),
    Row(product_id=3, product_name="Coffee Mug", category="Kitchen",     brand="Ikea",     color="White"),
    Row(product_id=4, product_name="Notebook",    category="Office",      brand="Moleskine", color="Red"),
    Row(product_id=5, product_name="Desk Lamp",   category="Office",      brand="Ikea",     color="Brass"),
])

dim_product.display()

## 3. Simulate incoming data batch

In [0]:
# =====================================================================
# SIMULATION OF INCOMING DATA (change feed from the source system)
# =====================================================================
# 60 change events (10x the original batch): one feed, but carrying
# multiple change points spread over several months.
#   * insert -> brand-new products (ids 6..22)
#   * update -> full new snapshot of the record; the SAME product_id can
#               appear many times with different change_dates (e.g. 1, 2,
#               3, 5, 6, 7, 8, 9, 10, 11, 20)
#   * delete -> product removed from the source (only the key matters)
# Every SCD type below consumes this SAME feed, each in its own way.
# NOTE: with multiple events per product, downstream logic typically has
# to pick the LATEST event per product (or replay events in date order)
# rather than blindly joining on product_id.

change_events = [
    # --- 2024-07-01: initial batch -----------------------------------
    # inserts: brand-new products
    (6,  "Mechanical Keyboard", "Electronics", "Logitech",       "Black",       "insert", "2024-07-01"),
    (7,  "Wall Clock",          "Home",        "Ikea",            "White",       "insert", "2024-07-01"),
    (8,  "Smart Watch",         "Electronics", "Garmin",          "Slate",       "insert", "2024-07-01"),
    (9,  "Yoga Mat",            "Sports",      "Decathlon",       "Blue",        "insert", "2024-07-01"),
    (10, "Blender",             "Kitchen",     "Philips",         "White",       "insert", "2024-07-01"),
    (11, "Office Chair",        "Office",      "Herman Miller",   "Black",       "insert", "2024-07-01"),
    (12, "Desk Organizer",      "Office",      "Knoll",            "Grey",        "insert", "2024-07-01"),
    (13, "Water Bottle",        "Sports",      "CamelBak",         "Cyan",        "insert", "2024-07-01"),
    (14, "Air Purifier",        "Home",        "Dyson",            "Silver",      "insert", "2024-07-01"),
    (15, "Gaming Mouse",        "Electronics", "Razer",            "Green",       "insert", "2024-07-01"),
    (16, "Standing Desk",       "Office",      "Varidesk",         "Oak",         "insert", "2024-07-01"),
    (17, "Curtains",            "Home",        "Zara Home",        "Beige",       "insert", "2024-07-01"),
    (18, "Coffee Grinder",      "Kitchen",     "Braun",            "Steel",       "insert", "2024-07-01"),
    (19, "Backpack",            "Travel",      "Samsonite",        "Red",         "insert", "2024-07-01"),
    (20, "Umbrella",            "Home",        "Totes",            "Navy",        "insert", "2024-07-01"),
    # updates: full new snapshot of existing products
    (1,  "Laptop",              "Computers",   "Acme",             "Silver",      "update", "2024-07-01"),
    (2,  "Headphones",          "Electronics", "Sony",             "Black",       "update", "2024-07-01"),
    (3,  "Coffee Mug",          "Home",        "Ikea",             "White",       "update", "2024-07-01"),

    # --- August: first wave of follow-up changes ----------------------
    (5,  "Desk Lamp",           "Lighting",    "Ikea",             "Brass",       "update", "2024-08-05"),
    (2,  "Headphones",          "Electronics", "Sony",             "Beige",       "update", "2024-08-12"),  # 2nd change for product 2
    (6,  "Mechanical Keyboard", "Electronics", "Razer",            "Black",       "update", "2024-08-20"),
    (1,  "Laptop",              "Computers",   "Acme",             "Graphite",    "update", "2024-08-25"),  # 2nd change for product 1
    (10, "Blender",             "Appliances",  "Philips",          "White",       "update", "2024-08-28"),
    (21, "Drone",               "Electronics", "DJI",              "Grey",        "insert", "2024-08-30"),  # late insert

    # --- September -----------------------------------------------------
    (12, None, None, None, None, "delete", "2024-09-10"),         # product 12 removed
    (2,  "Headphones",          "Audio",       "Sony",             "Beige",       "update", "2024-09-15"),  # 3rd change for product 2
    (7,  "Wall Clock",          "Decor",       "Ikea",             "White",       "update", "2024-09-18"),
    (3,  "Coffee Mug",          "Home",        "Ikea",             "Ivory",       "update", "2024-09-22"),  # 2nd change for product 3
    (22, "Bluetooth Speaker",   "Electronics", "Bose",             "Black",       "insert", "2024-09-25"),
    (5,  "Desk Lamp",           "Lighting",    "Philips",          "Brass",       "update", "2024-09-28"),  # 2nd change for product 5
    (1,  "Laptop Pro",          "Computers",   "Acme",             "Graphite",    "update", "2024-09-30"),  # renamed
    (8,  "Smart Watch",         "Electronics", "Garmin",           "Black",       "update", "2024-09-30"),

    # --- October -------------------------------------------------------
    (10, "Blender",             "Appliances",  "Philips",          "Cream",       "update", "2024-10-05"),
    (4,  "Notebook",            "Office",      "Moleskine",        "Black",       "update", "2024-10-08"),
    (20, "Umbrella",            "Home",        "Totes",            "Green",       "update", "2024-10-12"),
    (17, "Curtains",            "Home",        "Zara Home",        "White",       "update", "2024-10-15"),
    (9,  "Yoga Mat",            "Sports",      "Decathlon",        "Purple",      "update", "2024-10-18"),
    (2,  "Headphones",          "Audio",       "Sony",             "Navy",        "update", "2024-10-22"),  # 4th change for product 2
    (13, "Water Bottle",        "Sports",      "CamelBak",         "Charcoal",    "update", "2024-10-25"),
    (14, "Air Purifier",        "Appliances",  "Dyson",            "Silver",      "update", "2024-10-28"),
    (4,  None, None, None, None, "delete", "2024-10-31"),         # updated earlier in Oct, removed at month end

    # --- November ------------------------------------------------------
    (6,  "Mechanical Keyboard", "Electronics", "Razer",            "White",       "update", "2024-11-04"),  # 2nd change for product 6
    (11, "Office Chair",        "Office",      "Herman Miller",    "Grey",        "update", "2024-11-08"),
    (1,  "Laptop Pro",          "Computers",   "Acer",             "Graphite",    "update", "2024-11-12"),  # 4th change for product 1
    (16, "Standing Desk",       "Office",      "Varidesk",         "Walnut",      "update", "2024-11-15"),
    (10, "Blender",             "Appliances",  "Vitamix",          "Cream",       "update", "2024-11-18"),  # 3rd change for product 10
    (14, None, None, None, None, "delete", "2024-11-20"),         # product 14 removed
    (19, "Backpack",            "Travel",      "Samsonite",        "Black",       "update", "2024-11-22"),
    (5,  "Desk Lamp",           "Lighting",    "Philips",          "Matte Black",  "update", "2024-11-25"),  # 3rd change for product 5
    (8,  "Smart Watch",         "Electronics", "Apple",            "Black",       "update", "2024-11-28"),  # 2nd change for product 8

    # --- December ------------------------------------------------------
    (2,  "Headphones",          "Audio",       "Sennheiser",       "Navy",        "update", "2024-12-02"),  # 5th change for product 2
    (7,  "Wall Clock",          "Decor",       "Ikea",             "Oak",         "update", "2024-12-05"),  # 2nd change for product 7
    (15, "Gaming Mouse",        "Electronics", "Razer",            "Pink",        "update", "2024-12-08"),
    (22, "Bluetooth Speaker",   "Electronics", "Sonos",            "Black",       "update", "2024-12-10"),
    (9,  "Yoga Mat",            "Sports",      "Lululemon",        "Purple",      "update", "2024-12-12"),  # 2nd change for product 9
    (3,  "Coffee Mug",          "Home",        "Bodum",            "Ivory",       "update", "2024-12-15"),  # 3rd change for product 3
    (18, "Coffee Grinder",      "Kitchen",     "Baratza",          "Steel",       "update", "2024-12-18"),
    (11, "Office Chair",        "Office",      "Steelcase",        "Grey",        "update", "2024-12-20"),  # 2nd change for product 11
    (17, None, None, None, None, "delete", "2024-12-22"),         # product 17 removed
    (20, "Umbrella",            "Home",        "Windbrella",       "Green",       "update", "2024-12-28"),  # 2nd change for product 20
]

changes = spark.createDataFrame(
    change_events,
    """product_id long, product_name string, category string,
       brand string, color string, change_type string, change_date string""",
).withColumn("change_date", F.to_date("change_date")).orderBy("change_date", "product_id")

display(changes)

## 4. Behaviour of each SCD Type

In [0]:
# =====================================================================
# TYPE 0 — RETAIN (ORIGINAL VALUE)
# =====================================================================
# * inserts -> applied (new rows added once, then never touched again)
# * updates -> IGNORED (original value kept forever)
# * deletes -> IGNORED (row stays; Type 0 never changes anything)

inserts = changes.filter(F.col("change_type") == "insert")

dim_product_t0 = dim_product.unionByName(
    inserts.select("product_id", "product_name", "category", "brand", "color")
)

display(dim_product_t0.orderBy("product_id"))

In [0]:
# =====================================================================
# TYPE 1 — OVERWRITE
# =====================================================================
# * inserts -> added as new rows
# * updates -> existing row overwritten in place (old value lost)
# * deletes -> row removed

inserts = changes.filter(F.col("change_type") == "insert")
updates = changes.filter(F.col("change_type") == "update")
deletes = changes.filter(F.col("change_type") == "delete").select("product_id")

upd = updates.select(
    "product_id",
    F.col("product_name").alias("new_name"),
    F.col("category").alias("new_category"),
    F.col("brand").alias("new_brand"),
    F.col("color").alias("new_color"),
)

dim_product_t1 = (
    dim_product
    .join(upd, "product_id", "left")
    .withColumn("product_name", F.coalesce("new_name", "product_name"))
    .withColumn("category", F.coalesce("new_category", "category"))
    .withColumn("brand", F.coalesce("new_brand", "brand"))
    .withColumn("color", F.coalesce("new_color", "color"))
    .drop("new_name", "new_category", "new_brand", "new_color")
    .join(deletes, "product_id", "left_anti")
    .unionByName(inserts.select("product_id", "product_name", "category", "brand", "color"))
)

display(dim_product_t1.orderBy("product_id"))

In [0]:
# =====================================================================
# TYPE 2 — ADD NEW ROW (FULL HISTORY)
# =====================================================================
# * inserts -> new current rows
# * updates -> old version closed (valid_to = change_date - 1, is_current = false)
#              + new version inserted with a fresh surrogate key
# * deletes -> old version closed, no replacement row

# --- Step 0: give the base dimension Type 2 bookkeeping columns ---
dim_product_t2 = (
    dim_product
    .withColumn("product_sk", (F.row_number().over(Window.orderBy("product_id")) + F.lit(100)).cast("long"))
    .withColumn("valid_from", F.to_date(F.lit("2024-01-01")))
    .withColumn("valid_to", F.to_date(F.lit("9999-12-31")))
    .withColumn("is_current", F.lit(True))
)

# --- Step 1: close every row hit by an update or a delete ---
expired = (
    dim_product_t2
    .join(
        changes.filter(F.col("change_type").isin("update", "delete")).select("product_id", "change_date"),
        "product_id",
        "left",
    )
    .withColumn("is_current", F.when(F.col("change_date").isNotNull(), F.lit(False)).otherwise(F.col("is_current")))
    .withColumn("valid_to", F.when(F.col("change_date").isNotNull(), F.date_sub(F.col("change_date"), 1)).otherwise(F.col("valid_to")))
    .drop("change_date")
)

# --- Step 2: create new versions for inserts and updates ---
max_sk = dim_product_t2.agg(F.max("product_sk")).collect()[0][0]

new_versions = (
    changes
    .filter(F.col("change_type").isin("insert", "update"))
    .withColumn("product_sk", (F.row_number().over(Window.orderBy("product_id")) + F.lit(max_sk)).cast("long"))
    .withColumn("valid_from", F.col("change_date"))
    .withColumn("valid_to", F.to_date(F.lit("9999-12-31")))
    .withColumn("is_current", F.lit(True))
    .drop("change_type", "change_date")
)

dim_product_t2_final = expired.unionByName(new_versions)

display(dim_product_t2_final.orderBy("product_id", "product_sk"))

display(dim_product_t2_final.filter(F.col("is_current") == True) \
        .orderBy("product_id", "product_sk") \
        .drop("is_current")
        )



In [0]:
# =====================================================================
# TYPE 3 — ADD NEW COLUMN (CURRENT + PREVIOUS)
# =====================================================================
# * inserts -> added as new rows (no previous value yet)
# * updates -> old value moved to previous_category, new value becomes current
# * deletes -> row removed

inserts = changes.filter(F.col("change_type") == "insert")
updates = changes.filter(F.col("change_type") == "update")
deletes = changes.filter(F.col("change_type") == "delete").select("product_id")

# --- Step 0: add the "previous value" column ---
dim_product_t3 = dim_product.withColumn("previous_category", F.lit(None).cast("string"))

# --- Step 1: apply updates: current category moves to previous_category ---
upd = updates.select("product_id", F.col("category").alias("new_category"))

dim_product_t3 = (
    dim_product_t3
    .join(upd, "product_id", "left")
    .withColumn("previous_category", F.when(F.col("new_category").isNotNull(), F.col("category")).otherwise(F.col("previous_category")))
    .withColumn("category", F.coalesce("new_category", "category"))
    .drop("new_category")
    .join(deletes, "product_id", "left_anti")
)

# --- Step 2: add inserts (previous_category still unknown) ---
dim_product_t3 = dim_product_t3.unionByName(
    inserts.select("product_id", "product_name", "category", "brand", "color")
    .withColumn("previous_category", F.lit(None).cast("string"))
)

display(dim_product_t3.orderBy("product_id"))

In [0]:
# =====================================================================
# TYPE 4 — MINI-DIMENSION
# =====================================================================
# Rapidly changing attributes (category, brand, color) are split out of
# the base dimension into a mini-dimension: ONE row per unique attribute
# combination. The base dimension keeps only stable attributes plus a
# reference (mini_sk) to the current group — that reference is
# overwritten Type 1-style on change. History lives in the fact table's
# mini_sk references, not in the dimension itself.
#
# * inserts -> new group(s) in the mini-dim + new base row pointing at them
# * updates -> base row re-pointed to the (possibly new) matching group
# * deletes -> base row removed; groups stay (historical facts still use them)

inserts = changes.filter(F.col("change_type") == "insert")
updates = changes.filter(F.col("change_type") == "update")
deletes = changes.filter(F.col("change_type") == "delete").select("product_id")

# --- Step 0: build the mini-dimension from every attribute combination ---
# (existing rows + everything the incoming batch will produce)
all_combos = (
    dim_product.select("category", "brand", "color")
    .unionByName(
        changes.filter(F.col("change_type").isin("insert", "update"))
        .select("category", "brand", "color")
    )
    .distinct()
)

mini_dim_t4 = all_combos.withColumn(
    "mini_sk", (F.row_number().over(Window.orderBy("category", "brand", "color")) + F.lit(1000)).cast("long")
)

# --- Step 1: base dimension = stable attributes + current group reference ---
base_dim_t4 = (
    dim_product.select("product_id", "product_name", "category", "brand", "color")
    .join(mini_dim_t4, ["category", "brand", "color"])
    .select("product_id", "product_name", "mini_sk")
)

# --- Step 2: apply updates (re-point the base row — Type 1 on the reference) ---
upd_sk = (
    updates.select("product_id", "category", "brand", "color")
    .join(mini_dim_t4, ["category", "brand", "color"])
    .select("product_id", F.col("mini_sk").alias("new_mini_sk"))
)

base_dim_t4_final = (
    base_dim_t4
    .join(upd_sk, "product_id", "left")
    .withColumn("mini_sk", F.coalesce("new_mini_sk", "mini_sk"))
    .drop("new_mini_sk")
    .join(deletes, "product_id", "left_anti")
)

# --- Step 3: apply inserts (new base rows pointing at their groups) ---
ins_sk = (
    inserts.select("product_id", "product_name", "category", "brand", "color")
    .join(mini_dim_t4, ["category", "brand", "color"])
    .select("product_id", "product_name", "mini_sk")
)

base_dim_t4_final = base_dim_t4_final.unionByName(ins_sk)

print("Mini-dimension (one row per unique attribute combination):")
display(mini_dim_t4.orderBy("mini_sk"))
print("Base dimension (stable attributes + current group reference):")
display(base_dim_t4_final.orderBy("product_id"))

In [0]:
# =====================================================================
# TYPE 5 — MINI-DIMENSION + TYPE 1 OUTRIGGER
# =====================================================================
# Type 4 (cell above) PLUS the base dimension carries a Type 1 copy of
# the key mini-dimension attributes as an "outrigger" — overwritten on
# change. This lets you group/filter by the CURRENT value of a rapidly
# changing attribute directly from the base dimension, while the
# mini-dimension preserves the full history via mini_sk references.
#
# Requires the Type 4 cell above to have been executed
# (uses base_dim_t4_final and mini_dim_t4).

# --- Outrigger: current group attributes copied into the base dimension ---
# (in a real warehouse these columns are stored and Type 1-overwritten;
#  here they are derived from the current mini_sk reference, same result)
base_dim_t5 = (
    base_dim_t4_final
    .join(mini_dim_t4, "mini_sk")
    .select(
        "product_id",
        "product_name",
        "mini_sk",
        F.col("category").alias("current_category"),
        F.col("brand").alias("current_brand"),
    )
)

display(base_dim_t5.orderBy("product_id"))
# Note: product 3 (Coffee Mug) and product 7 (Wall Clock) share mini_sk —
# both point at the same (Home, Ikea, White) attribute group.

In [0]:
# =====================================================================
# TYPE 6 — HYBRID (TYPE 2 HISTORY + TYPE 1 CURRENT-VALUE COLUMN)
# =====================================================================
# Full history exactly like Type 2 (reusing dim_product_t2_final from the
# Type 2 cell above), PLUS a current_category column that carries the
# LATEST value on EVERY historical row — so old facts can also be grouped
# by the CURRENT attribute value, while the original columns still show
# the value as it was at each point in time.
#
# Requires the Type 2 cell above to have been executed (uses dim_product_t2_final).

# --- Step 1: latest category per product (from the current versions) ---
current_values = (
    dim_product_t2_final
    .filter(F.col("is_current"))
    .select("product_id", F.col("category").alias("current_category"))
)

# --- Step 2: stamp it onto every row, old versions included ---
dim_product_t6_final = dim_product_t2_final.join(current_values, "product_id", "left")

display(dim_product_t6_final.orderBy("product_id", "product_sk"))
# Note: product 3 shows category=Kitchen on the old row and Home on the new,
# but current_category=Home on BOTH rows. Deleted product 4 has no current
# version, so its historical row keeps current_category = null.

In [0]:
# =====================================================================
# TYPE 7 — DUAL TYPE 1 + TYPE 2 (TWO PHYSICAL TABLES)
# =====================================================================
# Both tables are maintained in parallel from the SAME change batch:
#  * dim_product_t1       (Type 1 cell)  -> current snapshot for dashboards
#  * dim_product_t2_final (Type 2 cell)  -> full history for point-in-time joins
#
# * inserts -> added to both tables
# * updates -> overwritten in the Type 1 table; new version row in the Type 2 table
# * deletes -> removed from the Type 1 table; last version closed in the
#              Type 2 table and kept as history
#
# Requires the Type 1 and Type 2 cells above to have been executed
# (uses dim_product_t1 and dim_product_t2_final).

print("Type 1 table — current snapshot (dashboards / latest value):")
display(dim_product_t1.orderBy("product_id"))
print("Type 2 table — full history (point-in-time fact joins):")
display(dim_product_t2_final.orderBy("product_id", "product_sk"))

In [0]:
# =====================================================================
# TYPE 8 — EMBEDDED HISTORY (SINGLE ROW WITH _scd ARRAY)
# =====================================================================
# One row per entity.  Current attribute values live in the regular
# columns (overwritten in place — Type 1 style).  The full version
# history is embedded in an _scd ARRAY<STRUCT> column — each element
# holds one version with its valid_from / valid_to and the attribute
# values that were current at that time.
#
# * inserts -> new row with _scd = [version 0]
# * updates -> current columns overwritten (Type 1) AND a new version
#              appended to _scd (version 1, 2, …)
# * deletes -> last version in _scd closed (valid_to set); row kept
#              with is_active = false for historical reference
#
# "Current" reporting never touches _scd (column pruning).
# History is reached via higher-order functions and/or explode().
# No is_current flag — the regular columns always hold the latest values.

# --- Step 0: unify original rows and change events into one stream ---
# Original products become "initial" events dated 2024-01-01.
initial = dim_product.select(
    "product_id", "product_name", "category", "brand", "color",
    F.lit("initial").alias("change_type"),
    F.to_date(F.lit("2024-01-01")).alias("event_date"),
)

all_events = initial.unionByName(
    changes.select(
        "product_id", "product_name", "category", "brand", "color",
        "change_type",
        F.col("change_date").alias("event_date"),
    )
)

# --- Step 1: build per-product version rows (non-delete events only) ---
# Each version gets a sequential number (0-based) and a valid_from /
# valid_to window derived from the NEXT event for the same product.
w = Window.partitionBy("product_id").orderBy("event_date")

versions = (
    all_events
    .filter(F.col("change_type") != "delete")
    .withColumn("version", F.row_number().over(w) - 1)
    .withColumn("next_date", F.lead("event_date").over(w))
    .withColumn("valid_from", F.col("event_date"))
    .withColumn(
        "valid_to",
        F.coalesce(F.date_sub(F.col("next_date"), 1), F.to_date(F.lit("9999-12-31"))),
    )
)

# --- Step 2: close the last version for deleted products ---
# If a product was deleted, its last version's valid_to is set to
# delete_date - 1 instead of 9999-12-31.
delete_dates = (
    changes
    .filter(F.col("change_type") == "delete")
    .select("product_id", F.col("change_date").alias("delete_date"))
)

versions = (
    versions
    .join(delete_dates, "product_id", "left")
    .withColumn(
        "valid_to",
        F.when(
            F.col("delete_date").isNotNull() & F.col("next_date").isNull(),
            F.date_sub(F.col("delete_date"), 1),
        ).otherwise(F.col("valid_to")),
    )
    .drop("next_date", "delete_date")
)

# --- Step 3: collect versions into _scd arrays (sorted by version) ---
scd_entries = versions.select(
    "product_id",
    F.struct(
        F.col("version").cast("long").alias("version"),
        F.col("valid_from").alias("valid_from"),
        F.col("valid_to").alias("valid_to"),
        F.col("product_name").alias("product_name"),
        F.col("category").alias("category"),
        F.col("brand").alias("brand"),
        F.col("color").alias("color"),
    ).alias("scd_entry"),
)

scd_arrays = (
    scd_entries
    .groupBy("product_id")
    .agg(F.collect_list("scd_entry").alias("_scd"))
    .withColumn("_scd", F.sort_array("_scd"))
)

# --- Step 4: pick the latest version's values as current columns ---
w_latest = Window.partitionBy("product_id").orderBy(F.col("version").desc())

current = (
    versions
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .select("product_id", "product_name", "category", "brand", "color")
)

# --- Step 5: determine is_active (false if the last event is a delete) ---
w_last = Window.partitionBy("product_id").orderBy(F.col("event_date").desc())

last_event = (
    all_events
    .withColumn("rn", F.row_number().over(w_last))
    .filter(F.col("rn") == 1)
    .select("product_id", F.col("change_type").alias("last_type"))
)

# --- Step 6: assemble the final Type 8 dimension ---
dim_product_t8_final = (
    current
    .join(scd_arrays, "product_id")
    .join(last_event, "product_id")
    .withColumn("is_active", F.col("last_type") != "delete")
    .drop("last_type")
    .orderBy("product_id")
)

display(dim_product_t8_final.orderBy("product_id"))

# Current snapshot — column pruning, _scd never touched
display(
    dim_product_t8_final
    .filter(F.col("is_active") == True)
    .drop("_scd", "is_active")
    .orderBy("product_id")
)

# History via explode — every version on its own row
display(
    dim_product_t8_final
    .select("product_id", "is_active", F.explode(F.col("_scd")).alias("v"))
    .select(
        "product_id", "is_active",
        F.col("v.version"), F.col("v.valid_from"), F.col("v.valid_to"),
        F.col("v.product_name").alias("hist_product_name"),
        F.col("v.category").alias("hist_category"),
        F.col("v.brand").alias("hist_brand"),
        F.col("v.color").alias("hist_color"),
    )
    .orderBy("product_id", "version")
)

## 5. Sample with the use of MERGE function in Data Lakehouse

In [0]:
# =====================================================================
# SETUP — MERGE examples (Delta Lake)
# =====================================================================
# MERGE INTO works on Delta tables only, so each example below first
# creates its target table in the `workspace.default` schema and then
# applies the incoming batch with one or more MERGE statements.
# The DataFrames from the cells above are registered as views so the
# SQL cells can use them as MERGE sources.

dim_product.createOrReplaceTempView("dim_product_v")
changes.createOrReplaceTempView("changes_v")

print("Views ready: dim_product_v, changes_v")

In [0]:
%sql
-- =====================================================================
-- SCD TYPE 1 — OVERWRITE via MERGE
-- =====================================================================
-- * delete -> WHEN MATCHED ... THEN DELETE
-- * update -> WHEN MATCHED ... THEN UPDATE SET (overwritten in place)
-- * insert -> WHEN NOT MATCHED ... THEN INSERT

CREATE OR REPLACE TABLE workspace.default.scd_t1_dim_product AS
SELECT * FROM dim_product_v;

MERGE INTO workspace.default.scd_t1_dim_product AS tgt
USING (
  SELECT product_id, product_name, category, brand, color, change_type, change_date
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
    FROM changes_v
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
  tgt.product_name = src.product_name,
  tgt.category     = src.category,
  tgt.brand        = src.brand,
  tgt.color        = src.color
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, category, brand, color)
  VALUES
  (src.product_id, src.product_name, src.category, src.brand, src.color);

SELECT * FROM workspace.default.scd_t1_dim_product ORDER BY product_id

In [0]:
%sql
-- =====================================================================
-- SCD TYPE 2 — ADD NEW ROW via MERGE
-- =====================================================================
-- MERGE can close the old version, but a new version is a brand-new
-- row that does not exist in the target yet — so the classic Delta
-- pattern is: one MERGE to expire old rows + one INSERT for new rows.

CREATE OR REPLACE TABLE workspace.default.scd_t2_dim_product AS
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id + 100 AS BIGINT) AS product_sk,   -- initial surrogate keys
  DATE'2024-01-01' AS valid_from,
  DATE'9999-12-31' AS valid_to,
  TRUE             AS is_current
FROM dim_product_v;

-- MERGE: close the current version of every updated / deleted product
MERGE INTO workspace.default.scd_t2_dim_product AS tgt
USING (
  SELECT product_id, change_date
  FROM (
    SELECT product_id, change_date,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date ASC) AS rn
    FROM changes_v WHERE change_type IN ('update', 'delete')
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
 AND tgt.is_current = true
WHEN MATCHED THEN UPDATE SET
  tgt.is_current = false,
  tgt.valid_to   = DATE_SUB(src.change_date, 1);

-- INSERT: new versions for updates, first versions for inserts
-- Unique surrogate keys and proper valid_from / valid_to windows
INSERT INTO workspace.default.scd_t2_dim_product
  (product_id, product_name, category, brand, color, product_sk, valid_from, valid_to, is_current)
WITH event_stream AS (
  SELECT product_id, product_name, category, brand, color, change_type, change_date,
    LEAD(change_date) OVER (PARTITION BY product_id ORDER BY change_date) AS next_date,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date) AS rn_asc
  FROM changes_v
  WHERE change_type IN ('insert', 'update', 'delete')
)
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id * 1000 + rn_asc + 99 AS BIGINT) AS product_sk,
  change_date AS valid_from,
  COALESCE(DATE_SUB(next_date, 1), DATE'9999-12-31') AS valid_to,
  CASE WHEN next_date IS NULL THEN TRUE ELSE FALSE END AS is_current
FROM event_stream
WHERE change_type IN ('insert', 'update');

SELECT * FROM workspace.default.scd_t2_dim_product ORDER BY product_id, product_sk

In [0]:
%sql
-- =====================================================================
-- SCD TYPE 3 — ADD NEW COLUMN via MERGE
-- =====================================================================
-- previous_category keeps the old value; category takes the new one.
-- UPDATE SET can read the target's pre-update value (tgt.category).

CREATE OR REPLACE TABLE workspace.default.scd_t3_dim_product AS
SELECT *, CAST(NULL AS STRING) AS previous_category
FROM dim_product_v;

MERGE INTO workspace.default.scd_t3_dim_product AS tgt
USING (
  SELECT product_id, product_name, category, brand, color, change_type, change_date
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
    FROM changes_v
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
  tgt.previous_category = tgt.category,   -- old value preserved
  tgt.category     = src.category,
  tgt.brand        = src.brand,
  tgt.color        = src.color
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, category, brand, color, previous_category)
  VALUES
  (src.product_id, src.product_name, src.category, src.brand, src.color, NULL);

SELECT * FROM workspace.default.scd_t3_dim_product ORDER BY product_id

In [0]:
%sql
-- =====================================================================
-- SCD TYPE 4 — MINI-DIMENSION via MERGE
-- =====================================================================
-- Two tables: a mini-dimension (one row per attribute combination) and
-- a base dimension holding stable attributes + the current mini_sk
-- reference. A single MERGE handles all three change kinds on the base
-- table: re-point the reference, delete the base row, insert new rows.

-- Mini-dimension: every combination that exists now OR arrives in the
-- batch. Surrogate keys are generated with DENSE_RANK here because
-- MERGE cannot generate new keys for rows it inserts.
CREATE OR REPLACE TABLE workspace.default.scd_t4_mini_dim AS
SELECT
  category, brand, color,
  CAST(1000 + DENSE_RANK() OVER (ORDER BY category, brand, color) AS BIGINT) AS mini_sk
FROM (
  SELECT category, brand, color FROM dim_product_v
  UNION
  SELECT category, brand, color FROM changes_v WHERE change_type IN ('insert', 'update')
) AS combos;

CREATE OR REPLACE TABLE workspace.default.scd_t4_base_dim AS
SELECT d.product_id, d.product_name, m.mini_sk
FROM dim_product_v AS d
JOIN workspace.default.scd_t4_mini_dim AS m
  ON d.category = m.category AND d.brand = m.brand AND d.color = m.color;

MERGE INTO workspace.default.scd_t4_base_dim AS tgt
USING (
  SELECT s.product_id, s.product_name, s.change_type, m.mini_sk
  FROM (
    SELECT product_id, product_name, category, brand, color, change_type
    FROM (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
      FROM changes_v
    )
    WHERE rn = 1
  ) AS s
  LEFT JOIN workspace.default.scd_t4_mini_dim AS m   -- LEFT: delete rows carry no attributes
    ON s.category = m.category AND s.brand = m.brand AND s.color = m.color
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET tgt.mini_sk = src.mini_sk
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, mini_sk)
  VALUES (src.product_id, src.product_name, src.mini_sk);

SELECT * FROM workspace.default.scd_t4_mini_dim ORDER BY mini_sk;

SELECT * FROM workspace.default.scd_t4_base_dim ORDER BY product_id

In [0]:
%sql
-- =====================================================================
-- SCD TYPE 5 — MINI-DIMENSION + TYPE 1 OUTRIGGER via MERGE
-- =====================================================================
-- Same two tables as Type 4, but the base dimension also carries a
-- Type 1 copy (outrigger) of the mini-dimension attributes — updated
-- in the same MERGE that re-points the mini_sk reference.

CREATE OR REPLACE TABLE workspace.default.scd_t5_mini_dim AS
SELECT
  category, brand, color,
  CAST(1000 + DENSE_RANK() OVER (ORDER BY category, brand, color) AS BIGINT) AS mini_sk
FROM (
  SELECT category, brand, color FROM dim_product_v
  UNION
  SELECT category, brand, color FROM changes_v WHERE change_type IN ('insert', 'update')
) AS combos;

CREATE OR REPLACE TABLE workspace.default.scd_t5_base_dim AS
SELECT
  d.product_id, d.product_name, m.mini_sk,
  m.category AS current_category,
  m.brand    AS current_brand
FROM dim_product_v AS d
JOIN workspace.default.scd_t5_mini_dim AS m
  ON d.category = m.category AND d.brand = m.brand AND d.color = m.color;

MERGE INTO workspace.default.scd_t5_base_dim AS tgt
USING (
  SELECT s.product_id, s.product_name, s.change_type, m.mini_sk, m.category, m.brand
  FROM (
    SELECT product_id, product_name, category, brand, color, change_type
    FROM (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
      FROM changes_v
    )
    WHERE rn = 1
  ) AS s
  LEFT JOIN workspace.default.scd_t5_mini_dim AS m
    ON s.category = m.category AND s.brand = m.brand AND s.color = m.color
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
  tgt.mini_sk          = src.mini_sk,
  tgt.current_category = src.category,
  tgt.current_brand    = src.brand
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, mini_sk, current_category, current_brand)
  VALUES
  (src.product_id, src.product_name, src.mini_sk, src.category, src.brand);

SELECT * FROM workspace.default.scd_t5_base_dim ORDER BY product_id

In [0]:
%sql
-- =====================================================================
-- SCD TYPE 6 — HYBRID (TYPE 2 + TYPE 1 CURRENT-VALUE COLUMN) via MERGE
-- =====================================================================
-- Three steps: close old versions (MERGE), add new versions (INSERT),
-- then stamp the latest category onto every historical row (second
-- MERGE). A MERGE source cannot be the target table itself, so the
-- current values are materialized into a small helper table first.

CREATE OR REPLACE TABLE workspace.default.scd_t6_dim_product AS
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id + 100 AS BIGINT) AS product_sk,
  DATE'2024-01-01' AS valid_from,
  DATE'9999-12-31' AS valid_to,
  TRUE AS is_current,
  category AS current_category      -- initially current == original
FROM dim_product_v;

-- MERGE 1: close the current version of every updated / deleted product
MERGE INTO workspace.default.scd_t6_dim_product AS tgt
USING (
  SELECT product_id, change_date
  FROM (
    SELECT product_id, change_date,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date ASC) AS rn
    FROM changes_v WHERE change_type IN ('update', 'delete')
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
 AND tgt.is_current = true
WHEN MATCHED THEN UPDATE SET
  tgt.is_current = false,
  tgt.valid_to   = DATE_SUB(src.change_date, 1);

-- INSERT: new versions (current_category already equals the new value)
INSERT INTO workspace.default.scd_t6_dim_product
  (product_id, product_name, category, brand, color, product_sk, valid_from, valid_to, is_current, current_category)
WITH event_stream AS (
  SELECT product_id, product_name, category, brand, color, change_type, change_date,
    LEAD(change_date) OVER (PARTITION BY product_id ORDER BY change_date) AS next_date,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date) AS rn_asc
  FROM changes_v
  WHERE change_type IN ('insert', 'update', 'delete')
)
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id * 1000 + rn_asc + 99 AS BIGINT) AS product_sk,
  change_date AS valid_from,
  COALESCE(DATE_SUB(next_date, 1), DATE'9999-12-31') AS valid_to,
  CASE WHEN next_date IS NULL THEN TRUE ELSE FALSE END AS is_current,
  category AS current_category
FROM event_stream
WHERE change_type IN ('insert', 'update');

-- Helper table: latest category per product (MERGE source != target)
CREATE OR REPLACE TABLE workspace.default.scd_t6_current_values AS
SELECT product_id, category
FROM workspace.default.scd_t6_dim_product
WHERE is_current = true;

-- MERGE 2: stamp the latest category onto EVERY historical row
MERGE INTO workspace.default.scd_t6_dim_product AS tgt
USING workspace.default.scd_t6_current_values AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED THEN UPDATE SET
  tgt.current_category = src.category;

SELECT * FROM workspace.default.scd_t6_dim_product ORDER BY product_id, product_sk

In [0]:
%sql
-- =====================================================================
-- SCD TYPE 7 — DUAL TYPE 1 + TYPE 2 via MERGE
-- =====================================================================
-- Two physical tables, each maintained with its own MERGE from the
-- SAME incoming batch: a current snapshot table (Type 1) and a full
-- history table (Type 2).

CREATE OR REPLACE TABLE workspace.default.scd_t7_dim_current AS
SELECT * FROM dim_product_v;

CREATE OR REPLACE TABLE workspace.default.scd_t7_dim_history AS
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id + 100 AS BIGINT) AS product_sk,
  DATE'2024-01-01' AS valid_from,
  DATE'9999-12-31' AS valid_to,
  TRUE AS is_current
FROM dim_product_v;

-- Type 1 side: overwrite in place / delete / insert
MERGE INTO workspace.default.scd_t7_dim_current AS tgt
USING (
  SELECT product_id, product_name, category, brand, color, change_type, change_date
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
    FROM changes_v
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
  tgt.product_name = src.product_name,
  tgt.category     = src.category,
  tgt.brand        = src.brand,
  tgt.color        = src.color
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, category, brand, color)
  VALUES (src.product_id, src.product_name, src.category, src.brand, src.color);

-- Type 2 side: close old versions ...
MERGE INTO workspace.default.scd_t7_dim_history AS tgt
USING (
  SELECT product_id, change_date
  FROM (
    SELECT product_id, change_date,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date ASC) AS rn
    FROM changes_v WHERE change_type IN ('update', 'delete')
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
 AND tgt.is_current = true
WHEN MATCHED THEN UPDATE SET
  tgt.is_current = false,
  tgt.valid_to   = DATE_SUB(src.change_date, 1);

-- ... and add the new versions
INSERT INTO workspace.default.scd_t7_dim_history
  (product_id, product_name, category, brand, color, product_sk, valid_from, valid_to, is_current)
WITH event_stream AS (
  SELECT product_id, product_name, category, brand, color, change_type, change_date,
    LEAD(change_date) OVER (PARTITION BY product_id ORDER BY change_date) AS next_date,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date) AS rn_asc
  FROM changes_v
  WHERE change_type IN ('insert', 'update', 'delete')
)
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id * 1000 + rn_asc + 99 AS BIGINT) AS product_sk,
  change_date AS valid_from,
  COALESCE(DATE_SUB(next_date, 1), DATE'9999-12-31') AS valid_to,
  CASE WHEN next_date IS NULL THEN TRUE ELSE FALSE END AS is_current
FROM event_stream
WHERE change_type IN ('insert', 'update');

SELECT * FROM workspace.default.scd_t7_dim_current ORDER BY product_id;

SELECT * FROM workspace.default.scd_t7_dim_history ORDER BY product_id, product_sk

In [0]:
%sql
-- =====================================================================
-- SCD TYPE 8 — EMBEDDED HISTORY via MERGE
-- =====================================================================
-- One row per entity.  Current values in regular columns (Type 1),
-- full version history embedded in an _scd ARRAY<STRUCT> column.
--
-- With multiple change events per product a single MERGE cannot
-- append multiple versions in one pass (each target row matches at
-- most one source row).  Instead we pre-compute the complete _scd
-- array per product, load it via CREATE OR REPLACE TABLE, then
-- demonstrate the incremental MERGE pattern for a single new change.

-- --- Build the complete final state from the source data ---
CREATE OR REPLACE TABLE workspace.default.scd_t8_dim_product AS
WITH all_events AS (
  SELECT product_id, product_name, category, brand, color,
         'initial'  AS change_type,
         DATE'2024-01-01' AS event_date
  FROM dim_product_v
  UNION ALL
  SELECT product_id, product_name, category, brand, color,
         change_type, change_date AS event_date
  FROM changes_v
),
versions AS (
  SELECT
    product_id, product_name, category, brand, color,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY event_date) - 1 AS version,
    event_date AS valid_from,
    LEAD(event_date) OVER (PARTITION BY product_id ORDER BY event_date) AS next_date
  FROM all_events
  WHERE change_type != 'delete'
),
closed AS (
  SELECT
    v.product_id, v.product_name, v.category, v.brand, v.color,
    v.version, v.valid_from,
    CASE
      WHEN d.delete_date IS NOT NULL AND v.next_date IS NULL
        THEN DATE_SUB(d.delete_date, 1)
      WHEN v.next_date IS NOT NULL
        THEN DATE_SUB(v.next_date, 1)
      ELSE DATE'9999-12-31'
    END AS valid_to
  FROM versions v
  LEFT JOIN (
    SELECT product_id, MAX(change_date) AS delete_date
    FROM changes_v WHERE change_type = 'delete'
    GROUP BY product_id
  ) d ON v.product_id = d.product_id
),
scd_arrays AS (
  SELECT
    product_id,
    ARRAY_SORT(COLLECT_LIST(
      named_struct(
        'version',      CAST(version AS BIGINT),
        'valid_from',    valid_from,
        'valid_to',      valid_to,
        'product_name',  product_name,
        'category',      category,
        'brand',         brand,
        'color',         color
      )
    )) AS _scd
  FROM closed
  GROUP BY product_id
),
current_vals AS (
  SELECT product_id, product_name, category, brand, color
  FROM (
    SELECT product_id, product_name, category, brand, color,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY version DESC) AS rn
    FROM closed
  ) WHERE rn = 1
),
active_flag AS (
  SELECT product_id,
         CASE WHEN change_type = 'delete' THEN false ELSE true END AS is_active
  FROM (
    SELECT product_id, change_type,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY event_date DESC) AS rn
    FROM all_events
  ) WHERE rn = 1
)
SELECT
  c.product_id, c.product_name, c.category, c.brand, c.color,
  s._scd, a.is_active
FROM current_vals c
JOIN scd_arrays   s ON c.product_id = s.product_id
JOIN active_flag  a ON c.product_id = a.product_id
ORDER BY c.product_id;

-- --- Incremental MERGE: add a single new product (demonstration) ---
-- For real incremental loads, process one change batch at a time
-- (at most one event per product per batch).
MERGE INTO workspace.default.scd_t8_dim_product AS tgt
USING (
  SELECT 99 AS product_id, 'Stylus Pen' AS product_name,
         'Electronics' AS category, 'Wacom' AS brand,
         'Black' AS color, DATE'2025-01-15' AS change_date
) AS src
  ON tgt.product_id = src.product_id
WHEN NOT MATCHED THEN INSERT
  (product_id, product_name, category, brand, color, _scd, is_active)
  VALUES (
    src.product_id, src.product_name, src.category, src.brand, src.color,
    ARRAY(named_struct(
      'version',      CAST(0 AS BIGINT),
      'valid_from',    src.change_date,
      'valid_to',      DATE'9999-12-31',
      'product_name',  src.product_name,
      'category',      src.category,
      'brand',         src.brand,
      'color',         src.color
    )),
    TRUE
  );

SELECT * FROM workspace.default.scd_t8_dim_product ORDER BY product_id;

-- History exploded — every version on its own row
SELECT
  product_id, is_active,
  hist.version, hist.valid_from, hist.valid_to,
  hist.product_name AS hist_product_name,
  hist.category     AS hist_category,
  hist.brand        AS hist_brand,
  hist.color        AS hist_color
FROM workspace.default.scd_t8_dim_product
LATERAL VIEW explode(_scd) scd_exploded AS hist
ORDER BY product_id, hist.version

## 6. Concept of SCD Type 8


### Basic characteristics of the Slowly Changing Dimensions type 8:

- name follows the next available number of the type — and, when rotated, also means "infinity"
- designed to take advantage of the "Data Lakehouse" architecture
- one row to store the current record and its history
    - simple navigation from the current context of a given entity
    - column pruning applied in most cases (getting current values)
    - simple way to get the previous value
    - previous values reached via higher-order functions and/or explode() to get the history
    - present row with current values (no flag `is_current`) - historical entries present in ARRAY `_scd`
- uses a `_scd` STRUCT type field that contains the history record (no separate history table object in Lakehouse)
    - each version of a record is an inner ARRAY element holding that version of the record (starting from version 0)
    - elasticity (each entity's version keeps its own schema with the values actual at that time)
- e.g., a ARRAY with the attributes, the change version, and the VALID_FROM / VALID_TO timestamp fields — all without creating separate rows (unlike, say, SCD Type 2)
- the schema of the `_scd` field is a free canvas:
    - a theoretical model can define recommended columns
    - "infinity" is achieved here through the flexibility of this schema — you can add whatever you consider needed
- the remaining columns in the table are the attributes that may change
- you can use higher-order functions to observe a given business phenomenon over time, according to the attribute values current at each moment
- "current" reporting happens without even touching the `_scd` column — each change is an append of a record to the `_scd` object plus an in-place update of the attributes directly in the table


In [0]:
# now recreate here the cell #1 but include the "SCD Type 8" characteristics and details (keept the format concise)

## 7. Point-in-time query — SCD Type 8

Explode the `_scd` array and filter on `valid_from` / `valid_to` to reconstruct the dimension table as it looked at any chosen date. No separate history table, no complex join — just column access into the embedded history.

In [0]:
# =====================================================================
# POINT-IN-TIME QUERY — "what did the dimension look like on date X?"
# =====================================================================
# SCD Type 8's _scd array makes point-in-time queries trivial:
# explode the _scd array, filter to the version whose valid_from /
# valid_to window contains the target date, and you get the exact
# attribute values every product had at that moment.
#
# We pick 2024-09-15 — a date that falls between several change waves,
# so products are at different versions (some at v0, some at v1, v2, …).

target_date = "2024-09-15"

point_in_time = (
    dim_product_t8_final
    .select("product_id", F.explode(F.col("_scd")).alias("v"))
    .filter(
        (F.col("v.valid_from") <= F.lit(target_date).cast("date"))
        & (F.col("v.valid_to") >= F.lit(target_date).cast("date"))
    )
    .select(
        "product_id",
        F.col("v.version"),
        F.col("v.product_name"),
        F.col("v.category"),
        F.col("v.brand"),
        F.col("v.color"),
        F.col("v.valid_from"),
        F.col("v.valid_to"),
    )
    .orderBy("product_id")
)

print(f"Dimension state on {target_date}:")
display(point_in_time)

# Slowly Changing Dimensions (SCD)

In dimensional modeling, a **slowly changing dimension** is a dimension whose attribute values change over time — slowly, not every transaction. Ralph Kimball defined six classic types (0–6); a seventh is commonly added in practice, and an eighth type extends the concept for the Data Lakehouse. Each type answers the question: *when an attribute value changes, what do we do with the old value?*

---

| Type | Name | Strategy | Year announced | Old value preserved? | Full history? | New objects created |
|------|------|---------|---------------|---------------------|---------------|--------------------|
| 0 | Retain / Original | Never change the value | 1996 (2008*) | ✅ (never overwritten) | ❌ | None — no schema or table changes |
| 1 | Overwrite | Replace old with new | 1996 | ❌ | ❌ | None — same table, same columns, in-place updates |
| 2 | Add New Row | Insert a new row, close out the old | 1996 | ✅ | ✅ | **New columns** added to the existing dimension: `valid_from`, `valid_to`, `is_current`, plus a **surrogate key** column (e.g. `product_sk`) |
| 3 | Add New Column | Keep old + new in separate columns | 1996 | ✅ (one prior version) | Partial (current + previous) | **New columns** added to the existing dimension (e.g. `previous_category`) — one per tracked attribute |
| 4 | Mini-Dimension | Rapidly changing attrs split into a separate dimension | 2002 | N/A (separate table) | ✅ (via mini-dim grouping) | **New table** (mini-dimension) for the rapidly-changing attributes, keyed by a surrogate key; base dimension gains a **new FK column** (e.g. `mini_sk`) pointing to it |
| 5 | Mini-Dimension + Type 1 Outrigger | Type 4 + Type 1 ref in base dim | 2002 | Partial | ✅ (via mini-dim) | Everything from Type 4 **plus new outrigger columns** copied into the base dimension (e.g. `current_category`, `current_brand`) carrying the latest mini-dim values |
| 6 | Hybrid (1 + 2) | Type 2 history + Type 1 overwrite for selected attrs | 2002 | Partial | ✅ | All Type 2 columns **plus a new current-value column** (e.g. `current_category`) stamped across every history row |
| 7 | Dual Type (1 + 2) | Maintain both a Type 1 table and a Type 2 table | 2013 | Partial | ✅ | **Second physical table** — a Type 1 snapshot table alongside the Type 2 history table (two tables maintained in parallel) |
| 8 | Embedded History (∞) | One row per entity; full history in `ARRAY<STRUCT>` `_scd` column | 2020s (Lakehouse) | ✅ | ✅ | **New column** `_scd` (`ARRAY<STRUCT>`) added to the existing dimension — no new tables; each array element holds one version with attributes, version number, and `valid_from`/`valid_to` |

---

*announced in 1996 with other concepts, but described in details in 2008.

## Detailed explanations

### Type 0 — Retain (Original Value)
Never modify the attribute — the value set at insert time is permanent, regardless of later changes in the source. Rarely used, but useful for immutable facts like a customer's original registration date or date of birth.

> **New objects:** None. The dimension table and its schema stay exactly as-is; no columns, tables, or lookup structures are added.
>
> **Announced:** Formally described by the Kimball Group in the third edition of *The Data Warehouse Toolkit* (2008), complementing the original three types.
> **Data Warehouse use case:** Preserving immutable attributes such as a customer's original registration date or date of birth that must never be modified.
> **Data Lakehouse use case:** Retaining original source-system landing timestamps in the Bronze layer for full audit traceability across the medallion architecture.

### Type 1 — Overwrite
Simply update the row in place. No history is kept — the old value is lost. This is the simplest and most common approach when only the current value matters (e.g., correcting a misspelled name).

> **New objects:** None. Same table, same columns — rows are simply overwritten in place.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Correcting a misspelled customer name or keeping the latest shipping address when only the current value matters.
> **Data Lakehouse use case:** Overwriting the latest record state in the Silver/Gold Delta tables via `MERGE` for downstream consumers that need only current values.

### Type 2 — Add New Row (Historical Tracking)
Insert a **new row** for the changed record and mark the previous row as expired. This preserves the full history of changes. Each row gets:
* **effective dates** (or timestamps): `valid_from` / `valid_to`
* a **current flag** (`is_current` = true/false)
* a **surrogate key** so every version has a unique primary key

This is the most widely used SCD type in data warehousing because it lets fact rows join to the dimension version that was active at the time of the fact.

> **New objects:** The **existing dimension table gains new columns** — at minimum a surrogate key (e.g. `product_sk`), `valid_from`, `valid_to`, and `is_current`. No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Tracking full customer address history so that fact rows join to the dimension version that was active at the time of each transaction.
> **Data Lakehouse use case:** Maintaining full version history in Delta tables using Change Data Feed (CDF) with `MERGE` SCD Type 2 logic for point-in-time analytics and incremental CDC pipelines.

### Type 3 — Add New Column
Add a second column to hold the prior value (e.g., `previous_category` alongside `category`). Only one level of history is kept, so this is suitable when you only need to compare the current value to the immediately preceding one.

> **New objects:** **New columns** added to the existing dimension table — one `previous_*` column per tracked attribute (e.g. `previous_category`). No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Comparing current vs. previous sales-territory assignment for an employee without the overhead of full history rows.
> **Data Lakehouse use case:** Lightweight one-level history tracking in a Silver Delta table when full SCD Type 2 is unnecessary but a before/after comparison is still required.

### Type 4 — Mini-Dimension
When certain attributes change **rapidly** (e.g., number of dependents, income band, credit score), they are split out of the main dimension into a separate *mini-dimension*. Each unique combination of those attributes becomes a row in the mini-dimension, keyed by a surrogate key. The base dimension references the current mini-dimension key (a Type 1 overwrite on that key). Fact rows then reference the mini-dimension surrogate key, capturing history without exploding the base dimension.

> **New objects:** A **new lookup table** (the mini-dimension) holding one row per unique attribute combination, keyed by its own surrogate key. The base dimension gains a **new foreign-key column** (e.g. `mini_sk`) referencing the mini-dimension.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) to handle rapidly changing attributes.
> **Data Warehouse use case:** Managing fast-changing customer demographics (income band, credit score, dependents) without exploding the base customer dimension.
> **Data Lakehouse use case:** Splitting volatile attributes into a separate Delta table keyed by surrogate key, referenced from the main dimension to keep the base table compact and queryable.

### Type 5 — Mini-Dimension + Type 1 Outrigger
A Type 4 mini-dimension, plus the **base dimension also carries a Type 1 copy** of the key mini-dimension attributes as an outrigger (overwritten on change). This lets you group or filter by the *current* value of a rapidly changing attribute directly from the base dimension, while the mini-dimension preserves the full history.

> **New objects:** Everything from Type 4 (new mini-dimension table + `mini_sk` FK on the base) **plus new outrigger columns** on the base dimension (e.g. `current_category`, `current_brand`) that carry a Type 1 copy of the latest mini-dimension attribute values.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as an extension of the Type 4 mini-dimension pattern.
> **Data Warehouse use case:** Allowing BI dashboards to filter by the current income band directly from the base dimension while the mini-dimension preserves full historical accuracy.
> **Data Lakehouse use case:** Adding current-value outrigger columns in the Gold layer so BI tools can query volatile attributes without joining to the mini-dimension Delta table.

### Type 6 — Hybrid (Type 1 + Type 2)
Combines Type 2 (full history rows with effective dates) **and** Type 1 (overwrite) for a subset of columns. Typical implementation: the dimension has history rows (Type 2), but also maintains a **current-value column** that is always overwritten to the latest value (Type 1) across *all* rows. This lets you group all historical facts by the current value of an attribute, while still preserving the value as it was at each point in time.

> **New objects:** All Type 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`) **plus a new current-value column** (e.g. `current_category`) added to the existing dimension table. No additional tables are created.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as a hybrid combining Types 1 and 2.
> **Data Warehouse use case:** Grouping all historical sales by the *current* product category while still preserving the category value as it was at each point in time.
> **Data Lakehouse use case:** Stamping a `current_*` column across every history row in a Delta table so a single Gold-layer query supports both current and point-in-time analytics.

### Type 7 — Dual Type 1 + Type 2
Maintain **two physical tables**: one Type 1 table (current state only, overwritten) and one Type 2 table (full history). Fact tables join to the Type 2 table for point-in-time accuracy, while the Type 1 table serves dashboards and queries that only need the latest snapshot. This trades extra storage for query simplicity.

> **New objects:** A **second physical table** — a Type 1 snapshot table alongside the Type 2 history table. The Type 2 table carries the SCD 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`); the Type 1 table mirrors the original dimension schema. Two tables must be maintained and kept in sync.
>
> **Announced:** Type 7 was formally described in 2013 by the Kimball Group. It was included in the 3rd edition of The Data Warehouse Toolkit the same year.
> **Data Warehouse use case:** Serving a Type 1 current-state table for fast dashboards and a Type 2 history table for accurate point-in-time fact joins from the same source feed.
> **Data Lakehouse use case:** Maintaining a current-state Delta table (feeding BI/SQL warehouses) alongside a full-history Delta table (feeding incremental CDC and audit pipelines), both refreshed from the same batch or streaming source.

### Type 8 — Embedded History (∞)
One row per entity stores **both current and historical data**. Current attribute values live in scalar columns (no `is_current` flag); the full version history is embedded in an `_scd` `ARRAY<STRUCT>` column where each element holds one version with its attributes, version number, and `valid_from` / `valid_to` timestamps. The name "8" also represents infinity (∞) when rotated, reflecting the unlimited schema flexibility of the embedded history field. Designed for the **Data Lakehouse** architecture: column pruning serves current-value queries without touching `_scd`, while higher-order functions and `explode()` access historical versions for point-in-time analysis. Each change appends a record to the `_scd` array and updates the scalar columns in place — no new rows, no separate history table.

> **New objects:** A **new column** `_scd` (`ARRAY<STRUCT>`) added to the existing dimension table. No additional tables are created — the full history lives inside the single row per entity.
>
> **Announced:** Designed for the Data Lakehouse architecture; the name follows the next available SCD type number and, when rotated, represents infinity (∞).
> **Data Warehouse use case:** Not traditionally applicable — this type was designed specifically for Lakehouse-native patterns.
> **Data Lakehouse use case:** Maintaining one row per entity in a Delta table with embedded history, enabling both current-value queries (via column pruning) and point-in-time analytics (via array functions) without a separate history table or multi-row joins.